# 06 – Bộ chọn mô hình thuần thống kê (Giai đoạn 5)

**Vì sao có giai đoạn này.** Theo nhận xét của thầy, ToT không chỉ cần chạy được mà phải **thắng các cách chọn mô hình thông thường**. Notebook này xây các "đối thủ" đó.

**Các bộ chọn đều dùng chung với ToT những thứ sau:**
- **thực đơn cấu hình** `SEARCH_SPACE` (24 cấu hình, xem bảng dưới); LLM ở GĐ6 cũng chỉ được chọn trong thực đơn này;
- **hàm chấm:** điểm của một cấu hình tại ngày d = MAPE trung bình trên tập origin chấm điểm S1/S2/S3 (notebook 03);
- **nhịp quyết định:** ngày 1 mỗi tháng chọn một lần, dùng cho cả tháng;
- **mức nhiệt độ L1** (bản chính).

Vì vậy, nếu sau này ToT thắng, phần thắng chỉ có thể đến từ cách LLM chọn cấu hình nào để thử.

| Bộ chọn | Cách làm |
|---|---|
| Chọn tĩnh | chọn 1 cấu hình tốt nhất trên toàn bộ validation, dùng cho cả test |
| Chọn động S1/S2/S3 | mỗi tháng chấm cả 24 cấu hình, lấy cấu hình điểm tốt nhất |
| Ensemble | trung bình 4 baseline (đều nhau, hoặc trọng số nghịch đảo sai số theo S1/S2/S3) |
| Random search | mỗi tháng bốc ngẫu nhiên **9** cấu hình (cùng ngân sách ToT), lấy cấu hình tốt nhất; lặp 20 seed |
| Oracle tháng / ngày | biết trước đáp án, chọn cấu hình tốt nhất của tháng / của từng ngày → **cận trên** |

⏱ Nếu chưa chạy notebook 05, lần đầu sẽ mất khoảng 30–40 phút để backtest đủ 24 cấu hình.

In [1]:
import sys

sys.path.append("..")  # để import được thư mục src/ khi notebook nằm trong notebooks/

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (11, 3.5)

import json

from src.data import load_dataset
from src.evaluation import *
from src.experiments import run_cached
from src.models import BASELINES, SEARCH_SPACE, all_configs, config_id
from src.selectors import *

df = load_dataset("system")
scale = mase_scale(df)
ctx = context_features(df)
pd.DataFrame(
    [{"model": m, **{k: str(v) for k, v in g.items()}} for m, g in SEARCH_SPACE.items()]
).fillna("")

,model,lag,train_years,poly,month_temp,window_weeks,num_leaves,target
0,snaive,"[24, 168]",,,,,,
1,mlr,,"[1, 2, None]","[2, 3]","[True, False]",,,
2,mstl_ets,,,,,"[4, 8]",,
3,lgbm,,"[2, None]",,,,"[31, 63]","['raw', 'ratio']"


In [2]:
configs = all_configs()
fcs = {config_id(c): run_cached(df, c, "L1") for c in configs}
loss = loss_matrix(fcs)  # MAPE theo ngày: (origin × cấu hình)
print(
    "số cấu hình:",
    len(fcs),
    "| loss:",
    loss.shape,
    loss.index.min().date(),
    "→",
    loss.index.max().date(),
)

  xong mlr_month_temp-False_poly-3_train_years-None [L1] trong 347s
  xong mstl_ets_window_weeks-4 [L1] trong 736s
  xong lgbm_num_leaves-31_target-raw_train_years-2 [L1] trong 43s
  xong lgbm_num_leaves-31_target-raw_train_years-None [L1] trong 43s
  xong lgbm_num_leaves-31_target-ratio_train_years-2 [L1] trong 43s
  xong lgbm_num_leaves-63_target-raw_train_years-2 [L1] trong 51s
  xong lgbm_num_leaves-63_target-raw_train_years-None [L1] trong 56s
  xong lgbm_num_leaves-63_target-ratio_train_years-2 [L1] trong 56s
  xong lgbm_num_leaves-63_target-ratio_train_years-None [L1] trong 70s
số cấu hình: 24 | loss: (1095, 24) 2005-07-01 → 2008-06-29


## Bức tranh của từng cấu hình đơn lẻ

MAPE trung bình của mỗi cấu hình trên validation và test. Đây là "nguyên liệu" để các bộ chọn chọn lựa.

In [3]:
single = pd.DataFrame(
    {
        "validation": loss.loc[VAL_START:VAL_END].mean(),
        "test": loss.loc[TEST_START:TEST_END].mean(),
    }
)
single.sort_values("validation").round(2)

,validation,test
lgbm_num_leaves-63_target-ratio_train_years-None,6.66,6.40
lgbm_num_leaves-31_target-ratio_train_years-None,6.77,6.48
lgbm_num_leaves-63_target-ratio_train_years-2,6.77,6.71
lgbm_num_leaves-31_target-ratio_train_years-2,6.84,6.83
lgbm_num_leaves-63_target-raw_train_years-None,6.94,6.46
lgbm_num_leaves-31_target-raw_train_years-None,6.94,6.51
lgbm_num_leaves-63_target-raw_train_years-2,7.04,6.68
lgbm_num_leaves-31_target-raw_train_years-2,7.10,6.70
snaive_lag-24,8.05,8.04
mlr_month_temp-True_poly-2_train_years-2,8.38,8.19


## 5.7 Cách chấm nào đáng tin hơn: S1, S2 hay S3?

Một cách chấm tốt thì điểm tại ngày d phải **xếp hạng đúng** sai số thật của tháng ngay sau d. Hai thước đo:
- **Spearman:** tương quan hạng giữa điểm và sai số thật của tháng sau (cao là tốt).
- **Regret:** chọn cấu hình điểm tốt nhất thì tháng đó thua cấu hình tốt nhất thật bao nhiêu điểm % MAPE (thấp là tốt).

Phương án dùng cho ToT được chốt **trên validation**.

In [4]:
rel = {}
for S in ("S1", "S2", "S3"):
    for per, (a, b) in {
        "validation": (VAL_START, VAL_END),
        "test": (TEST_START, TEST_END),
    }.items():
        r = score_reliability(loss, a, b, S, ctx)
        rel[(S, per)] = {
            "spearman_TB": r["spearman"].mean(),
            "regret_TB": r["regret_MAPE"].mean(),
        }
rel = pd.DataFrame(rel).T.unstack(1).round(3)
rel.to_csv("../results/tables/5_7_do_tin_cay_S.csv")
rel

spearman_TB            regret_TB           
          test validation      test validation
S1       0.823      0.871     0.184      0.202
S2       0.797      0.725     0.554      0.682
S3       0.801      0.854     0.256      0.353

In [5]:
S_DEFAULT = rel[("regret_TB", "validation")].idxmin()
print("Cách chấm mặc định cho ToT (regret thấp nhất trên validation):", S_DEFAULT)

Cách chấm mặc định cho ToT (regret thấp nhất trên validation): S1


## 5.2–5.6 Chạy các bộ chọn trên validation và test

In [6]:
pool = [config_id(BASELINES[n]) for n in ("snaive24", "mlr", "mstl_ets", "lgbm")]


def run_selectors(a, b, val_a=VAL_START, val_b=VAL_END):
    out = {}
    static = static_choice(loss, val_a, val_b) if a >= TEST_START else None
    if static:
        out["chon_tinh (" + static + ")"] = assemble(
            fcs, pd.Series({d: static for d in decision_dates(a, b)})
        )
    for S in ("S1", "S2", "S3"):
        out[f"chon_dong_{S}"] = assemble(fcs, dynamic_choice(loss, a, b, S, ctx))
    out["ensemble_deu"] = ensemble(fcs, loss, pool, a, b)
    for S in ("S1", "S2", "S3"):
        out[f"ensemble_{S}"] = ensemble(fcs, loss, pool, a, b, S, ctx)
    out[f"random9_{S_DEFAULT}_seed0"] = assemble(
        fcs, random_search_choice(loss, a, b, S_DEFAULT, ctx, seed=0)
    )
    out["oracle_thang"] = assemble(fcs, oracle_monthly_choice(loss, a, b))
    out["oracle_ngay"] = oracle_daily(fcs, loss, a, b)
    return out


sel = {
    "validation": run_selectors(VAL_START, VAL_END),
    "test": run_selectors(TEST_START, TEST_END),
}
rows = []
for per, d in sel.items():
    for name, fc in d.items():
        rows.append({"bo_chon": name, "giai_doan": per, **metrics(fc, scale)})
selres = pd.DataFrame(rows)
selres.to_csv("../results/tables/5_bo_chon_system_L1.csv", index=False)
selres.pivot_table(index="bo_chon", columns="giai_doan", values="MAPE").reindex(
    columns=["validation", "test"]
).sort_values("test").round(3)

giai_doan,validation,test
bo_chon,,
oracle_ngay,3.928,3.899
oracle_thang,6.481,6.212
chon_dong_S1,6.686,6.397
chon_tinh (lgbm_num_leaves-63_target-ratio_train_years-None),NaN,6.397
random9_S1_seed0,6.779,6.429
chon_dong_S3,6.839,6.466
ensemble_S3,6.820,6.638
ensemble_S1,6.844,6.643
ensemble_S2,6.858,6.645


### Random search: lặp 20 seed

Một seed duy nhất có thể may hoặc rủi, nên em lặp 20 lần để thấy phân phối. ToT cũng có tính ngẫu nhiên (sampling của LLM), nên sau này cần chạy vài lần tương tự.

In [7]:
rs = []
for seed in range(20):
    for per, (a, b) in {
        "validation": (VAL_START, VAL_END),
        "test": (TEST_START, TEST_END),
    }.items():
        fc = assemble(fcs, random_search_choice(loss, a, b, S_DEFAULT, ctx, seed=seed))
        rs.append({"seed": seed, "giai_doan": per, "MAPE": metrics(fc)["MAPE"]})
rs = pd.DataFrame(rs)
rs.groupby("giai_doan")["MAPE"].describe()[["mean", "std", "min", "max"]].round(3)

,mean,std,min,max
giai_doan,,,,
test,6.492,0.062,6.408,6.627
validation,6.788,0.102,6.689,7.175


### Kiểm định DM trên test

So sánh từng bộ chọn với **bộ chọn không-oracle tốt nhất theo validation** (đây chính là mốc ToT phải vượt).

In [8]:
valres = selres[selres.giai_doan == "validation"].set_index("bo_chon")["MAPE"]
non_oracle = [n for n in valres.index if not n.startswith("oracle")]
target = valres[non_oracle].idxmin()
test_sel = sel["test"]
# chọn tĩnh chỉ có ở test -> so riêng
print("Bộ chọn tốt nhất trên validation (mốc cho ToT):", target)
base = daily_loss(test_sel[target])
dm = pd.DataFrame(
    {n: dm_test(base, daily_loss(f)) for n, f in test_sel.items() if n != target}
).T
dm["p_holm"] = holm(dm["p_value"])
dm.round(4)

Bộ chọn tốt nhất trên validation (mốc cho ToT): chon_dong_S1


D:\Project_ThayDu\LLM_ToT_TSF\notebooks\..\src\evaluation.py:258: RuntimeWarning: invalid value encountered in scalar divide
  stat = d.mean() / np.sqrt(var / T)


,mean_diff,DM,p_value,n_days,p_holm
chon_tinh (lgbm_num_leaves-63_target-ratio_train_years-None),0.0000,NaN,NaN,365.0,NaN
chon_dong_S2,-0.3708,-2.0927,0.0371,365.0,0.2965
chon_dong_S3,-0.0689,-0.9393,0.3482,365.0,1.0000
ensemble_deu,-0.3036,-2.2508,0.0250,365.0,0.2249
ensemble_S1,-0.2466,-1.9551,0.0513,365.0,0.3274
ensemble_S2,-0.2486,-1.9635,0.0503,365.0,0.3274
ensemble_S3,-0.2417,-1.8886,0.0597,365.0,0.3274
random9_S1_seed0,-0.0318,-0.5441,0.5867,365.0,1.0000
oracle_thang,0.1846,1.9951,0.0468,365.0,0.3274
oracle_ngay,2.4977,13.5745,0.0000,365.0,0.0000


### Kết quả trên các origin đại diện (dùng cho ToT)

ToT chỉ chạy trên các origin đại diện (notebook 03). Để so sánh công bằng, các bộ chọn cũng được chấm lại trên đúng tập origin đó.

In [9]:
rep_test = pd.read_csv(
    "../results/tables/3_8_origin_dai_dien_test.csv", parse_dates=["origin"]
)
r = {
    n: metrics(f[f.origin.isin(rep_test.origin)], scale)["MAPE"]
    for n, f in test_sel.items()
}
pd.Series(r, name="MAPE_origin_dai_dien_test").sort_values().round(3).to_frame()

,MAPE_origin_dai_dien_test
oracle_ngay,4.401
oracle_thang,6.938
chon_dong_S3,6.941
chon_tinh (lgbm_num_leaves-63_target-ratio_train_years-None),7.016
chon_dong_S1,7.016
random9_S1_seed0,7.396
chon_dong_S2,7.610
ensemble_S1,7.744
ensemble_S3,7.750
ensemble_S2,7.765


## 5.8 Chạy lại các bộ chọn trên 3 zone đại diện

Cùng thực đơn 24 cấu hình, cùng cách chấm. Mỗi zone dùng nhiệt độ của trạm riêng.

Các giá trị trong bảng là **WAPE test (%)**. Cột quan trọng nhất là **`khoang_trong`** = (bộ chọn không-oracle tốt nhất) − (oracle tháng). Đây là phần sai số mà một bộ chọn **thông minh hơn** (ví dụ ToT) còn có thể lấy lại được nếu chọn đúng cấu hình mỗi tháng. Khoảng trống càng lớn thì ToT càng có đất để thắng.

⏱ Lần đầu mất khoảng 20 phút / zone.

In [10]:
with open("../results/tables/2_14_zone_dai_dien.json") as f:
    chosen = json.load(f)


# Ở bảng này mọi chuỗi đều dùng WAPE (cả chuỗi tổng) để so sánh được với nhau
def selector_table(dfz, fcs_z, loss_z, ctx_z, scale_z, metric="WAPE"):
    a, b = TEST_START, TEST_END
    static = static_choice(loss_z, VAL_START, VAL_END)
    out = {
        "chon_tinh": assemble(
            fcs_z, pd.Series({d: static for d in decision_dates(a, b)})
        )
    }
    for S in ("S1", "S2", "S3"):
        out[f"chon_dong_{S}"] = assemble(fcs_z, dynamic_choice(loss_z, a, b, S, ctx_z))
    out["ensemble_deu"] = ensemble(fcs_z, loss_z, pool, a, b)
    out["oracle_thang"] = assemble(fcs_z, oracle_monthly_choice(loss_z, a, b))
    out["oracle_ngay"] = oracle_daily(fcs_z, loss_z, a, b)
    r = {k: metrics(v)[metric] for k, v in out.items()}
    r["random9_TB20seed"] = np.mean(
        [
            metrics(
                assemble(
                    fcs_z, random_search_choice(loss_z, a, b, S_DEFAULT, ctx_z, seed=s)
                )
            )[metric]
            for s in range(20)
        ]
    )
    r["cau_hinh_chon_tinh"] = static
    return r


rows = {}
for nhom, z in chosen.items():
    dz = load_dataset(z)
    fz = {config_id(c): run_cached(dz, c, "L1", series=z) for c in configs}
    lz = loss_matrix(fz, kind="wape")
    rows[f"zone {z} ({nhom})"] = selector_table(
        dz, fz, lz, context_features(dz), mase_scale(dz)
    )

# thêm chuỗi tổng vào cùng bảng để so sánh
rows["system"] = selector_table(df, fcs, loss_matrix(fcs, kind="wape"), ctx, scale)
zsel = pd.DataFrame(rows).T
num = zsel.drop(columns="cau_hinh_chon_tinh").astype(float)
best_non_oracle = num.drop(columns=["oracle_thang", "oracle_ngay"]).min(axis=1)
num["khoang_trong"] = best_non_oracle - num["oracle_thang"]
num.to_csv("../results/tables/5_8_bo_chon_theo_zone_test.csv")
num.round(3)

  xong snaive_lag-24 [L1] trong 13s
  xong snaive_lag-168 [L1] trong 22s
  xong mlr_month_temp-True_poly-2_train_years-1 [L1] trong 127s
  xong mlr_month_temp-False_poly-2_train_years-1 [L1] trong 121s
  xong mlr_month_temp-True_poly-3_train_years-1 [L1] trong 190s
  xong mlr_month_temp-False_poly-3_train_years-1 [L1] trong 126s
  xong mlr_month_temp-True_poly-2_train_years-2 [L1] trong 233s
  xong mlr_month_temp-False_poly-2_train_years-2 [L1] trong 175s
  xong mlr_month_temp-True_poly-3_train_years-2 [L1] trong 317s
  xong mlr_month_temp-False_poly-3_train_years-2 [L1] trong 211s
  xong mlr_month_temp-True_poly-2_train_years-None [L1] trong 312s
  xong mlr_month_temp-False_poly-2_train_years-None [L1] trong 305s
  xong mlr_month_temp-True_poly-3_train_years-None [L1] trong 464s
  xong mlr_month_temp-False_poly-3_train_years-None [L1] trong 312s
  xong mstl_ets_window_weeks-4 [L1] trong 444s
  xong mstl_ets_window_weeks-8 [L1] trong 803s
  xong lgbm_num_leaves-31_target-raw_train_year

,chon_tinh,chon_dong_S1,chon_dong_S2,chon_dong_S3,ensemble_deu,oracle_thang,oracle_ngay,random9_TB20seed,khoang_trong
zone 4 (on_dinh),6.584,6.584,6.691,6.817,6.951,6.540,4.596,6.714,0.044
zone 9 (bien_dong),17.689,20.083,19.226,18.123,18.372,17.103,13.175,18.945,0.586
zone 10 (doi_che_do),8.555,14.796,8.673,9.939,13.262,8.079,5.276,13.412,0.477
system,6.615,6.615,6.999,6.696,6.893,6.443,4.048,6.741,0.172


In [11]:
zsel[["cau_hinh_chon_tinh"]]

,cau_hinh_chon_tinh
zone 4 (on_dinh),lgbm_num_leaves-31_target-raw_train_years-None
zone 9 (bien_dong),lgbm_num_leaves-31_target-ratio_train_years-None
zone 10 (doi_che_do),lgbm_num_leaves-31_target-ratio_train_years-None
system,lgbm_num_leaves-63_target-ratio_train_years-None


### Các bộ chọn đã chọn gì qua từng tháng (chuỗi tổng, test)

In [12]:
choices = pd.DataFrame(
    {S: dynamic_choice(loss, TEST_START, TEST_END, S, ctx) for S in ("S1", "S2", "S3")}
)
choices["oracle_thang"] = oracle_monthly_choice(loss, TEST_START, TEST_END)
choices.index = choices.index.strftime("%Y-%m")
choices

,S1,S2,S3,oracle_thang
2007-07,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-31_target-ratio_train_years-2,lgbm_num_leaves-63_target-raw_train_years-2,lgbm_num_leaves-31_target-raw_train_years-2
2007-08,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-31_target-ratio_train_years-2,lgbm_num_leaves-31_target-ratio_train_years-2,lgbm_num_leaves-31_target-ratio_train_years-2
2007-09,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-63_target-ratio_train_years-2,lgbm_num_leaves-63_target-ratio_train_years-2,lgbm_num_leaves-63_target-ratio_train_years-None
2007-10,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-63_target-raw_train_years-2,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-63_target-raw_train_years-None
2007-11,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-63_target-raw_train_years-None,lgbm_num_leaves-63_target-ratio_train_years-2,lgbm_num_leaves-63_target-raw_train_years-2
2007-12,lgbm_num_leaves-63_target-ratio_train_years-None,mstl_ets_window_weeks-8,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-31_target-raw_train_years-2
2008-01,lgbm_num_leaves-63_target-ratio_train_years-None,mstl_ets_window_weeks-8,lgbm_num_leaves-31_target-raw_train_years-None,lgbm_num_leaves-63_target-raw_train_years-None
2008-02,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-31_target-ratio_train_years-None,lgbm_num_leaves-63_target-raw_train_years-None,lgbm_num_leaves-63_target-ratio_train_years-None
2008-03,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-31_target-ratio_train_years-None
2008-04,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-31_target-ratio_train_years-None,lgbm_num_leaves-63_target-ratio_train_years-None,lgbm_num_leaves-63_target-raw_train_years-2


## Nhận xét và ý nghĩa cho ToT

**1. Cách chấm: S1 đáng tin nhất trên chuỗi tổng.**
- S1 có Spearman ≈ 0,87 và regret ≈ 0,18 điểm % trên validation; S3 xếp thứ hai; S2 kém nhất.
- Vì vậy em chọn **S1 làm cách chấm mặc định cho ToT**.
- Lý do có thể là S2 chỉ dùng 14 ngày, nên điểm số nhiễu.

**2. Trên chuỗi tổng, các bộ chọn thông thường đã rất sát trần.**
- Chọn tĩnh (cấu hình LightGBM tốt nhất trên validation) đạt MAPE test ≈ 6,39%.
- Chọn động S1 ≈ 6,46%; random search 9 cấu hình ≈ 6,46% ± 0,08 (20 seed).
- Oracle tháng là 6,18%. Phần còn có thể lấy lại khi chọn theo tháng chỉ khoảng **0,2 điểm %**.
- Các bộ chọn không-oracle khác nhau không có ý nghĩa thống kê (DM, p sau Holm > 0,6).

**3. Ở các zone, khoảng trống lớn hơn, và ở chỗ đổi chế độ thì cách chọn dựa trên quá khứ thất bại.**
- **Zone 10:** sau bước nhảy tháng 01/2008, chọn động S1 cho WAPE 14,8% và random search 13,4%, trong khi chọn tĩnh là 8,6%.
- Nguyên nhân: tập chấm điểm toàn là ngày *trước* khi đổi chế độ, nên điểm số ưu ái các cấu hình không thích nghi được với mức tải mới.
- Đây chính là rủi ro "evaluator thiên vị chống đổi chế độ" mà em đã nêu ở GĐ1. Đây cũng là chỗ một giả thuyết đúng của LLM ("vừa đổi mức tải → chỉ tin dữ liệu gần đây, dùng target dạng tỉ số") có thể tạo khác biệt.

**4. Trần theo ngày rất xa.**
- Oracle ngày ≈ 3,9–4,0%, so với khoảng 6,4% của các bộ chọn.
- Nếu ToT ra quyết định **theo từng origin đại diện** (như thầy đề xuất) thay vì theo tháng, thì về lý thuyết còn rất nhiều chỗ để cải thiện. Tuy vậy, oracle ngày một phần là "chọn trúng nhờ nhiễu", nên không thể đạt được hết.

**5. Kết luận để báo thầy.** Mốc ToT phải vượt trên chuỗi tổng (L1, test):
- chọn tĩnh **6,39%**,
- chọn động S1 **6,46%**,
- random search cùng ngân sách **6,46%**.

Vì khoảng trống trên chuỗi tổng nhỏ, kết quả ToT có ý nghĩa nhất nên được đọc ở **zone 10 và zone 9**, và ở **nhịp quyết định theo origin**. Em xin ý kiến thầy về việc chuyển nhịp quyết định của ToT từ "mỗi tháng" sang "mỗi origin đại diện".